# From One Neuron to a Disciplined Network
### A House Prices deep dive: depth, training, and regularization

**Trap → Fix → Build → Trap → Fix**, the same arc as the slide deck. Run cells top to bottom.


In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv('/home/claude/train.csv')
df.head()


,GrLivArea,OverallQual,YearBuilt,TotalBsmtSF,GarageCars,LotArea,NeighborhoodTier,FullBath,SalePrice
0,1748,8,1958,845,3,8071,1,2,352097
1,1431,10,1893,835,2,14396,1,2,368238
2,1824,10,2007,809,1,17839,1,2,412832
3,2262,8,1933,498,0,11802,2,3,390046
4,1383,1,1908,949,1,10331,3,2,237006


## Part 1 — Trap: One Neuron Isn't Enough

A single neuron computes `y_hat = w·x + b` then (optionally) squashes it with an activation.
With NO hidden layers, that's just **linear regression wearing a neural-network costume.**

We built `SalePrice` with a real interaction term baked in:
`OverallQual * GrLivArea * 0.08` — quality and size *amplify each other*. A bigger house
matters more when it's also high quality. A straight line through feature space cannot
represent "these two things multiply," only "these two things add."

Let's prove it: fit one linear neuron and watch it fail systematically, not randomly.


In [2]:
features = ['GrLivArea','OverallQual','YearBuilt','TotalBsmtSF','GarageCars',
            'LotArea','NeighborhoodTier','FullBath']
X = df[features].values.astype('float32')
y = df['SalePrice'].values.astype('float32').reshape(-1,1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

x_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(y_train)
X_train_s = x_scaler.transform(X_train)
X_test_s  = x_scaler.transform(X_test)
y_train_s = y_scaler.transform(y_train)
y_test_s  = y_scaler.transform(y_test)

Xtr = torch.tensor(X_train_s, dtype=torch.float32)
ytr = torch.tensor(y_train_s, dtype=torch.float32)
Xte = torch.tensor(X_test_s, dtype=torch.float32)
yte = torch.tensor(y_test_s, dtype=torch.float32)


In [3]:
# ONE neuron == nn.Linear(in_features, 1) with no hidden layer, no activation
single_neuron = nn.Linear(len(features), 1)
opt = torch.optim.Adam(single_neuron.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

history_linear = []
for epoch in range(300):
    opt.zero_grad()
    pred = single_neuron(Xtr)
    loss = loss_fn(pred, ytr)
    loss.backward()
    opt.step()
    history_linear.append(loss.item())

with torch.no_grad():
    test_pred_s = single_neuron(Xte)
    test_mse_linear = loss_fn(test_pred_s, yte).item()
    # back to dollars for an intuitive error
    test_pred_dollars = y_scaler.inverse_transform(test_pred_s.numpy())
    rmse_dollars = np.sqrt(np.mean((test_pred_dollars - y_test)**2))

print(f"Single-neuron test RMSE: ${rmse_dollars:,.0f}")


Single-neuron test RMSE: $28,995


**Diagnose the failure, don't just note it.** Bucket houses by `OverallQual` and look at
the *signed* residual (predicted − actual) within each bucket. A linear model's errors should
be flat-ish across quality if it's just noisy. If the errors trend with quality, that's the
fingerprint of a missed interaction — the model is structurally wrong, not just imprecise.


In [4]:
with torch.no_grad():
    resid = (test_pred_dollars.flatten() - y_test.flatten())
qual_bucket = pd.cut(X_test[:,1], bins=[0,3,6,8,10])
resid_df = pd.DataFrame({'qual_bucket': qual_bucket, 'residual': resid})
print(resid_df.groupby('qual_bucket')['residual'].mean())


qual_bucket
(0, 3]     -6864.768555
(3, 6]     14100.863281
(6, 8]      1836.842896
(8, 10]   -15793.074219
Name: residual, dtype: float32


You should see a **trend**, not noise — the residual drifts from one sign to another as
quality rises. That's the interaction term leaking through. A single neuron has no mechanism
to say "multiply these two," only "add them up with fixed weights." This is the Trap.


## Part 2 — Fix & Build: Stacking Layers Gives the Network Hands

A hidden layer + a nonlinear activation (ReLU) lets the network build its **own** interaction
terms. Each hidden neuron is still just `w·x + b`, but ReLU's kink means the *combination* of
many neurons can bend into curves and crossings a single line never could.

**Architecture for this problem:**
```
Input (8 features)
   -> Linear(8, 32)  -> ReLU
   -> Linear(32, 16) -> ReLU
   -> Linear(16, 1)            (raw SalePrice, scaled)
```

Forward pass, written out so nothing is hidden:
`h1 = ReLU(X @ W1 + b1)` → `h2 = ReLU(h1 @ W2 + b2)` → `y_hat = h2 @ W3 + b3`


In [5]:
class HousePriceNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layer1 = nn.Linear(n_features, 32)
        self.layer2 = nn.Linear(32, 16)
        self.output = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        h1 = self.relu(self.layer1(x))   # first bend
        h2 = self.relu(self.layer2(h1))  # second bend, built from the first
        return self.output(h2)           # linear combination of bent features

net = HousePriceNet(len(features))
opt = torch.optim.Adam(net.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

train_hist, test_hist = [], []
for epoch in range(400):
    net.train()
    opt.zero_grad()
    pred = net(Xtr)
    loss = loss_fn(pred, ytr)
    loss.backward()
    opt.step()
    train_hist.append(loss.item())

    net.eval()
    with torch.no_grad():
        test_hist.append(loss_fn(net(Xte), yte).item())

    if epoch % 100 == 0:
        print(f"epoch {epoch:4d}  train MSE {train_hist[-1]:.4f}  test MSE {test_hist[-1]:.4f}")


epoch    0  train MSE 1.1417  test MSE 1.0955
epoch  100  train MSE 0.0155  test MSE 0.0178


epoch  200  train MSE 0.0119  test MSE 0.0165


epoch  300  train MSE 0.0108  test MSE 0.0170


In [6]:
net.eval()
with torch.no_grad():
    test_pred_s2 = net(Xte)
    test_pred_dollars2 = y_scaler.inverse_transform(test_pred_s2.numpy())
    rmse_dollars2 = np.sqrt(np.mean((test_pred_dollars2 - y_test)**2))

print(f"Single neuron   test RMSE: ${rmse_dollars:,.0f}")
print(f"Hidden-layer NN test RMSE: ${rmse_dollars2:,.0f}")

resid2 = (test_pred_dollars2.flatten() - y_test.flatten())
resid_df2 = pd.DataFrame({'qual_bucket': qual_bucket, 'residual': resid2})
print(resid_df2.groupby('qual_bucket')['residual'].mean())


Single neuron   test RMSE: $28,995
Hidden-layer NN test RMSE: $10,403
qual_bucket
(0, 3]    -1013.880310
(3, 6]     1445.537476
(6, 8]      798.301880
(8, 10]   -4380.794434
Name: residual, dtype: float32


Check the residual-by-quality table again: it should flatten out toward zero across
buckets. That flattening *is* the interaction term being learned — not declared, learned,
from data, via depth + nonlinearity. That's the entire conceptual payoff of Part 2.


## Part 3 — Trap: A Network That Memorizes

Give the network *too much* capacity and *too little* discipline, and it stops learning the
relationship and starts memorizing the 1,168 training rows — including their noise. Watch the
train/test loss curves diverge as training continues.


In [7]:
big_net = nn.Sequential(
    nn.Linear(len(features), 256), nn.ReLU(),
    nn.Linear(256, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 1)
)
opt = torch.optim.Adam(big_net.parameters(), lr=0.01)

overfit_train, overfit_test = [], []
for epoch in range(600):
    big_net.train()
    opt.zero_grad()
    loss = loss_fn(big_net(Xtr), ytr)
    loss.backward()
    opt.step()
    overfit_train.append(loss.item())
    big_net.eval()
    with torch.no_grad():
        overfit_test.append(loss_fn(big_net(Xte), yte).item())

print(f"final train MSE: {overfit_train[-1]:.4f}   final test MSE: {overfit_test[-1]:.4f}")
print(f"gap (test - train): {overfit_test[-1]-overfit_train[-1]:.4f}")


final train MSE: 0.0011   final test MSE: 0.0205
gap (test - train): 0.0194


The train loss keeps falling while test loss flattens or *rises*. That growing gap is
overfitting, visible as a number, not a vibe. The network has enough capacity (256-wide
layers, ~1,160 training rows) to memorize idiosyncrasies of specific houses rather than the
general pattern. This is the Trap.


## Part 4 — Fix: Regularization, Three Different Mechanisms

These three techniques fight overfitting in genuinely different ways — worth contrasting,
not just listing:

- **L2 (weight decay):** adds `λ·Σw²` to the loss. Penalizes *large* weights, pushing the
  network toward many small, distributed weights instead of a few huge, brittle ones.
- **L1:** adds `λ·Σ|w|`. Pushes weights toward *exactly zero* — a form of automatic feature
  selection inside the network, not just shrinkage.
- **Dropout:** randomly zeroes a fraction of neurons *during training only*. No single neuron
  can become indispensable, because it might vanish on the next batch — forces redundant,
  robust representations.
- **Batch normalization:** normalizes each layer's activations to mean 0 / std 1 per batch,
  then lets the network learn a new scale and shift. Stabilizes the *distribution* of
  activations layer to layer, which lets you train deeper nets with higher learning rates.

Same big architecture as Part 3, now disciplined:


In [8]:
class RegularizedNet(nn.Module):
    def __init__(self, n_features, p_drop=0.3):
        super().__init__()
        self.fc1 = nn.Linear(n_features, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.out = nn.Linear(128, 1)
        self.drop = nn.Dropout(p_drop)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.drop(self.relu(self.bn1(self.fc1(x))))
        x = self.drop(self.relu(self.bn2(self.fc2(x))))
        x = self.relu(self.bn3(self.fc3(x)))
        return self.out(x)

reg_net = RegularizedNet(len(features))
# L2 lives right inside the optimizer as weight_decay
opt = torch.optim.Adam(reg_net.parameters(), lr=0.01, weight_decay=1e-3)
l1_lambda = 1e-4

reg_train, reg_test = [], []
for epoch in range(600):
    reg_net.train()
    opt.zero_grad()
    pred = reg_net(Xtr)
    base_loss = loss_fn(pred, ytr)
    l1_penalty = sum(p.abs().sum() for p in reg_net.parameters())
    loss = base_loss + l1_lambda * l1_penalty
    loss.backward()
    opt.step()
    reg_train.append(base_loss.item())

    reg_net.eval()
    with torch.no_grad():
        reg_test.append(loss_fn(reg_net(Xte), yte).item())

print(f"Overfit net   -> final gap: {overfit_test[-1]-overfit_train[-1]:.4f}")
print(f"Regularized   -> final gap: {reg_test[-1]-reg_train[-1]:.4f}")


Overfit net   -> final gap: 0.0194
Regularized   -> final gap: -0.0005


## Wrap-up: Compare All Three Models in Dollars


In [9]:
reg_net.eval()
with torch.no_grad():
    reg_pred_dollars = y_scaler.inverse_transform(reg_net(Xte).numpy())
    reg_rmse = np.sqrt(np.mean((reg_pred_dollars - y_test)**2))

big_net.eval()
with torch.no_grad():
    overfit_pred_dollars = y_scaler.inverse_transform(big_net(Xte).numpy())
    overfit_rmse = np.sqrt(np.mean((overfit_pred_dollars - y_test)**2))

summary = pd.DataFrame({
    'Model': ['Single neuron (linear)', 'Hidden-layer NN', 'Big NN, no regularization', 'Big NN, regularized'],
    'Test RMSE ($)': [rmse_dollars, rmse_dollars2, overfit_rmse, reg_rmse]
})
summary


,Model,Test RMSE ($)
0,Single neuron (linear),28994.767578
1,Hidden-layer NN,10402.894531
2,"Big NN, no regularization",11367.704102
3,"Big NN, regularized",13158.847656


**Takeaways to carry into class:**
1. Depth + nonlinearity = the network's way of discovering interaction terms you didn't hand-engineer.
2. Capacity without discipline memorizes; the train/test gap is the receipt.
3. L1, L2, dropout, and batch norm all fight overfitting, but through different mechanisms —
   sparsity, shrinkage, redundancy, and stabilized activation distributions, respectively.
